In [ ]:
# =========================================================
# INSTALLS
# =========================================================
!pip install --quiet "qdrant-client[fastembed]>=1.14.2" sentence-transformers fastembed nltk transformers pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 17.7 MB/s eta 0:00:00


In [ ]:
# =========================================================
# IMPORTS
# =========================================================
import os, json, shutil
import numpy as np
import nltk

from nltk.tokenize import sent_tokenize

from fastembed import TextEmbedding, SparseTextEmbedding
from qdrant_client import QdrantClient, models

# ================= NEW =================
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
# =======================================

# =========================================================
# NLTK
# =========================================================
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# =========================================================
# CONFIG
# =========================================================
DISEASES_JSON = "diseases_en.json"
PESTS_JSON    = "pests_en.json"

SECTION_FOR_DISEASE = "Visual Symptoms"
SECTION_FOR_PEST    = "Visual Symptoms"

EMBED_MODEL = "all-MiniLM-L6-v2"

COLLECTION = "VDB_DP"
IMAGE_COLLECTION = "VDB_DP_IMAGES"   # ================= NEW =================

QDRANT_STORAGE = "./qdrant_store"

# ================= NEW =================
PROTOTYPE_DATASET = "/content/RicePrototypeDataset"

CLIP_MODEL = "openai/clip-vit-base-patch32"
# =======================================

In [ ]:
!unzip RicePrototypeDataset.zip

Archive:  RicePrototypeDataset.zip
   creating: RicePrototypeDataset/Bacterial Leaf Blight/
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_3210.jpeg  
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_3295.jpeg  
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_8761.jpeg  
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_8788.jpeg  
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_8817.jpeg  
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_8843.jpeg  
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_8882.jpeg  
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_8928.jpeg  
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_IMG_20220407_072833.jpg  
  inflating: RicePrototypeDataset/Bacterial Leaf Blight/BLB_IMG_20220407_073930.jpg  
   creating: RicePrototypeDataset/Bacterial Panicle Blight/
  inflating: RicePrototypeDataset/Bacterial Panicle Blight/3353.jpeg  
  inflating: RicePrototypeDat

In [ ]:
# =========================================================
# LOAD CLIP
# =========================================================

clip_model = CLIPModel.from_pretrained(CLIP_MODEL)

processor = CLIPProcessor.from_pretrained(CLIP_MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
# =========================================================
# IMAGE EMBEDDING
# =========================================================
def encode_image_clip(path):

    img = Image.open(path).convert("RGB")

    inputs = processor(
        images=img,
        return_tensors="pt"
    )

    with torch.no_grad():

        vision_outputs = clip_model.vision_model(
            pixel_values=inputs["pixel_values"]
        )

        pooled = vision_outputs.pooler_output

        feat = clip_model.visual_projection(
            pooled
        )

    feat = torch.nn.functional.normalize(
        feat,
        p=2,
        dim=-1,
    )

    feat = feat.squeeze().cpu().numpy()

    return feat.tolist()

In [ ]:
def extract_target_sentences(data, entity_key, target_section_name, entity_type):
    """
    data: list of entries read from JSON
    entity_key: "disease" or "pest" (the field name in JSON entry for the name)
    target_section_name: e.g., "Visual Symptoms"
    entity_type: "disease" or "pest"
    Returns list of dicts: {type, name, section, text}
    """
    out = []
    for entry in data:
        entity_name = entry.get(entity_key)
        if not entity_name:
            continue
        for sec in entry.get("sections", []):
            if sec.get("name") == target_section_name:
                text = sec.get("text", "").strip()
                if not text:
                    continue
                # split into sentences
                for s in sent_tokenize(text):
                    s = s.strip()
                    if s:
                        out.append({
                            "type": entity_type,
                            "name": entity_name,
                            "section": sec.get("name"),
                            "text": s
                        })
    return out




In [ ]:
def load_json_safe(path):
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

def build_corpus_from_json(diseases_json, pests_json,
                           section_disease=SECTION_FOR_DISEASE,
                           section_pest=SECTION_FOR_PEST):
    """
    Returns:
      corpus_meta: list of payload dicts for each sentence {type, name, section, text}
      corpus_sentences: list of sentence strings (same order)
      name_to_entry: mapping name -> full JSON entry (all sections)
    """
    diseases = load_json_safe(diseases_json)
    pests = load_json_safe(pests_json)

    name_to_entry = {}
    if diseases:
        for d in diseases:
            nm = d.get("disease")
            if nm:
                # save full entry (all sections)
                name_to_entry[nm] = {"type":"disease", "sections": d.get("sections", [])}
    if pests:
        for p in pests:
            nm = p.get("pest")
            if nm:
                name_to_entry[nm] = {"type":"pest", "sections": p.get("sections", [])}

    # Only extract Visual Symptoms sentences for corpus_meta
    d_sentences = extract_target_sentences(diseases or [], "disease", section_disease, "disease")
    p_sentences = extract_target_sentences(pests or [], "pest", section_pest, "pest")

    corpus_meta = d_sentences + p_sentences
    corpus_sentences = [c["text"] for c in corpus_meta]

    print(f"✅ Built corpus: {len(corpus_sentences)} sentences, {len(name_to_entry)} entities.")
    return corpus_meta, corpus_sentences, name_to_entry


In [ ]:
corpus_meta, corpus_sentences, name_to_entry = build_corpus_from_json(DISEASES_JSON, PESTS_JSON)

✅ Built corpus: 88 sentences, 29 entities.


In [ ]:
def build_qdrant_and_save(
    corpus_meta,
    corpus_sentences,
    name_to_entry,
    qdrant_path="./qdrant_store",
    collection_name="hybrid_collection",
    dense_model_name="sentence-transformers/all-MiniLM-L6-v2",
    sparse_model_name="Qdrant/bm25"
):
    """
    Build local Qdrant collection with dense + sparse embeddings (FastEmbed).
    Compatible with Qdrant >= 1.14.2 and [fastembed].
    """

    dense_encoder = TextEmbedding(model_name=dense_model_name)
    sparse_encoder = SparseTextEmbedding(model_name=sparse_model_name)

    dense_vector_name = "dense"
    sparse_vector_name = "sparse"

    # Clean store if exists
    if os.path.exists(qdrant_path):
        shutil.rmtree(qdrant_path)

    client = QdrantClient(path=qdrant_path)

    # For cloud db creation
    # client = QdrantClient(
    #     url="",
    #     api_key="",
    # )

    dense_dim = dense_encoder.embedding_size

    # --- Create hybrid collection ---
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            dense_vector_name: models.VectorParams(
                size=dense_dim,
                distance=models.Distance.COSINE,
            )
        },
        sparse_vectors_config={
            sparse_vector_name: models.SparseVectorParams(modifier=models.Modifier.IDF),
        },
    )

    # --- Encode embeddings ---
    print("Encoding dense embeddings...")
    dense_embeddings = list(dense_encoder.embed(corpus_sentences))

    print("Encoding sparse embeddings...")
    sparse_embeddings = list(sparse_encoder.embed(corpus_sentences))

    # --- Upload points ---
    print("Uploading points to Qdrant...")
    points = []
    for idx, (dense_vec, sparse_vec, meta) in enumerate(
        zip(dense_embeddings, sparse_embeddings, corpus_meta)
    ):
        points.append(
            models.PointStruct(
                id=idx,
                vector={
                    dense_vector_name: dense_vec,
                    sparse_vector_name: sparse_vec.as_object(),
                },
                payload=meta,
            )
        )

    client.upsert(collection_name=collection_name, points=points)
    print(f"✅ Uploaded {len(points)} hybrid (dense+sparse) points to '{collection_name}'")

    # =====================================================
    # IMAGE COLLECTION
    # =====================================================

    client.create_collection(
        collection_name=IMAGE_COLLECTION,

        vectors_config=models.VectorParams(
            size=512,
            distance=models.Distance.COSINE,
        ),
    )

    image_points = []
    img_idx = 0

    print("Uploading prototype images...")

    for class_name in os.listdir(PROTOTYPE_DATASET):
        class_dir = os.path.join(PROTOTYPE_DATASET, class_name)

        if not os.path.isdir(class_dir):
            continue

        for img_name in os.listdir(class_dir):

            if not img_name.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):
                continue

            img_path = os.path.join(class_dir, img_name)
            vec = encode_image_clip(img_path)

            ctype = "disease"

            if class_name in ["Rice Hispa", "Stem Borer"]:
                ctype = "pest"
            elif class_name == "Healthy":
                ctype = "healthy"


            image_points.append(
                models.PointStruct(
                    id=img_idx,
                    vector=vec,
                    payload={
                        "type": ctype,
                        "name": class_name,
                        "image_path": img_path,
                    }
                )
            )

            img_idx += 1

    client.upsert(
        collection_name=IMAGE_COLLECTION,
        points=image_points
    )

    print(f"✅ Uploaded {len(image_points)} prototype images")
    # =====================================================

    # --- Separate name_to_entry (optional) ---
    if name_to_entry:
        name_coll = "name_to_entry"
        if client.collection_exists(name_coll):
            client.delete_collection(name_coll)

        client.create_collection(
            collection_name=name_coll,
            vectors_config=models.VectorParams(
                size=dense_dim,
                distance=models.Distance.COSINE,
            ),
        )

        name_points = []
        for idx, (name, entry) in enumerate(name_to_entry.items()):
            vec = list(dense_encoder.embed([name]))[0]

            section_texts = []
            visual_symptoms_text = None
            for section in entry.get("sections", []):
                section_texts.append(f"{section['name']}: {section['text']}")

                # Extract only Visual Symptoms
                if section['name'].lower() == "visual symptoms":
                    visual_symptoms_text = section['text']

                combined_text = "\n".join(section_texts)

            name_points.append(
                models.PointStruct(
                    id=idx,
                    vector=vec,
                    payload={
                      "name": name,
                      "type": entry.get("type", "unknown"),
                      "sections_text": combined_text,
                      "visual_symptoms": visual_symptoms_text,
                    },
                )
            )

        client.upsert(collection_name=name_coll, points=name_points)
        print(f"✅ Uploaded {len(name_points)} name_to_entry records")

    return client


In [ ]:
if os.path.exists(QDRANT_STORAGE):
       shutil.rmtree(QDRANT_STORAGE)
       print("🗑️ Removed existing Qdrant storage.")

🗑️ Removed existing Qdrant storage.


In [ ]:
# -------------- Qdrant --------------
corpus_meta, corpus_sentences, name_to_entry = build_corpus_from_json(DISEASES_JSON, PESTS_JSON)

if corpus_meta is None or name_to_entry is None:
  raise FileNotFoundError("No JSON corpus available. Can't build Qdrant.")

client = build_qdrant_and_save(corpus_meta, corpus_sentences, name_to_entry)


✅ Built corpus: 88 sentences, 29 entities.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Encoding dense embeddings...
Encoding sparse embeddings...
Uploading points to Qdrant...
✅ Uploaded 88 hybrid (dense+sparse) points to 'hybrid_collection'
Uploading prototype images...
✅ Uploaded 219 prototype images
✅ Uploaded 29 name_to_entry records


In [ ]:
# for cloud only
from qdrant_client.models import PayloadSchemaType

client.create_payload_index(
    collection_name="name_to_entry",
    field_name="name",
    field_schema=PayloadSchemaType.KEYWORD,
)

/tmp/ipykernel_7839/1544552750.py:3: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
!zip -r qdrant_store_CLIP.zip ./qdrant_store

  adding: qdrant_store/ (stored 0%)
  adding: qdrant_store/.lock (stored 0%)
  adding: qdrant_store/meta.json (deflated 80%)
  adding: qdrant_store/collection/ (stored 0%)
  adding: qdrant_store/collection/name_to_entry/ (stored 0%)
  adding: qdrant_store/collection/name_to_entry/storage.sqlite (deflated 41%)
  adding: qdrant_store/collection/hybrid_collection/ (stored 0%)
  adding: qdrant_store/collection/hybrid_collection/storage.sqlite (deflated 31%)
  adding: qdrant_store/collection/VDB_DP_IMAGES/ (stored 0%)
  adding: qdrant_store/collection/VDB_DP_IMAGES/storage.sqlite (deflated 53%)
